# Import File and Packages

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import pandas as pd

file_path = '/content/drive/MyDrive/550 Project/Regression Versions/simon/IBM Opensource Employee data about turnover.xlsx'
df = pd.read_excel(file_path)

print(f"Successfully loaded data from: {file_path}")
display(df.head())

Successfully loaded data from: /content/drive/MyDrive/550 Project/Regression Versions/simon/IBM Opensource Employee data about turnover.xlsx


,Age,Attrition,Attrition_1,Attrition_0,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,1,0,Travel_Rarely,1102,Sales,1,2,Life Sciences,...,1,80,0,8,0,1,6,4,0,5
1,49,No,0,1,Travel_Frequently,279,Research & Development,8,1,Life Sciences,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,1,0,Travel_Rarely,1373,Research & Development,2,2,Other,...,2,80,0,7,3,3,0,0,0,0
3,33,No,0,1,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,...,3,80,0,8,3,3,8,7,3,0
4,27,No,0,1,Travel_Rarely,591,Research & Development,2,1,Medical,...,4,80,1,6,3,3,2,2,2,2


## Package Installation

In [56]:
import os
import json
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Script Overview

**ONE controlled logistic regression (statsmodels, logit)**

## Purpose
- Produce a single, defensible **driver model** with **p-values + odds ratios (OR)**.
- Keep the workflow **short, rerunnable, and easy to interpret** for the team.

## Design choices (important)
- **One representation per concept** (avoid duplicate/collinear “same story” variables).
- Include **controls** so comparisons are fair across employees.

## What we run (high-level flow)
1) **Load data** → create `AttritionFlag` (Yes=1/No=0) and `OverTimeFlag` (Yes=1/No=0)  
2) **Within-family “horse race” selection** (scientific grouping):  
   - For each concept (e.g., *tenure*, *compensation*, *satisfaction*), test candidate variables and keep **1 (or 2 if clearly justified)**  
3) **Fit one final controlled logistic regression** using:
   - **Selected driver variables** (from the horse race)
   - **Categorical controls** (Department, JobRole, JobLevel, BusinessTravel, MaritalStatus, Gender)
4) **Print clean results tables** (analyst + executive-friendly)

## Interpretation (how to read results)
- **OR > 1** → associated with **higher** attrition risk  
- **OR < 1** → associated with **lower** attrition risk (“protective”)  
- These are **associations**, not proven causation.

## Notes
- This Step 2 file is for **driver evidence / inference** (p-values + OR).  
- We **do not use CV** in this locked driver model (CV is for predictive validation/segmentation steps).

## Outputs
- Prints:
  - Horse race tables (what we tested + what we kept)
  - Full model summary (appendix-style)
  - Clean ranked table of key terms (OR + p-value)

# Feature Preparation

In [57]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names for consistent downstream references."""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace("  +", " ", regex=True)
    )
    return df


def apply_feature_definitions(df: pd.DataFrame) -> pd.DataFrame:
    """Create minimal derived fields needed for the locked driver model."""
    df = standardize_columns(df)

    # AttritionFlag: 1 if Attrition == "Yes", else 0
    if "Attrition" not in df.columns:
        raise ValueError("Expected column 'Attrition' not found in dataset.")
    df["AttritionFlag"] = (df["Attrition"].astype(str).str.strip().str.lower() == "yes").astype(int)

    # Ensure key string columns are clean (safe no-ops if already clean)
    for c in ["OverTime", "BusinessTravel", "Department", "JobRole", "MaritalStatus", "Gender"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

    return df


# Output Control

In [58]:
# -----------------
# Output control (Colab-friendly)
# -----------------
# If you run this in Google Colab, set OUTPUT_DIR to your Drive folder, e.g.:
OUTPUT_DIR = "/content/drive/MyDrive/550 Project/Regression Versions/simon"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data Load

In [59]:
df = apply_feature_definitions(df)

# Drop constant columns if present
for col in ["Over18", "EmployeeCount", "StandardHours"]:
    if col in df.columns:
        df = df.drop(columns=[col])

if "AttritionFlag" not in df.columns:
    raise ValueError("AttritionFlag not found. Ensure feature_definitions creates it.")

# Core drivers + controls
numeric_vars = [
    "Age",
    "YearsAtCompany",
    "MonthlyIncome",
    "DistanceFromHome",
    "StockOptionLevel",
    "PercentSalaryHike",
    "TrainingTimesLastYear",
    "YearsSinceLastPromotion",
    "JobInvolvement",
    "JobSatisfaction",
    "EnvironmentSatisfaction",
    "WorkLifeBalance",
]

categorical_controls = [
    "Department",
    "JobRole",
    "JobLevel",
    "BusinessTravel",
    "MaritalStatus",
    "Gender",
]

# Overtime flag (Yes/No -> 1/0)
if "OverTime" in df.columns:
    df["OverTimeFlag"] = (df["OverTime"].astype(str).str.lower() == "yes").astype(int)
else:
    raise ValueError("OverTime column not found in dataset.")

# Features Selections

In [60]:
"""Scientific grouping + within-family horse race (to justify keep vs drop)"""

# Define variable families with candidate variables (only include if present in df/model_df)
FAMILIES = {
    "Tenure & Experience": [
        "YearsAtCompany",
        "TotalWorkingYears",
        "YearsInCurrentRole",
        "YearsWithCurrManager",
    ],
    "Compensation & Rewards": [
        "MonthlyIncome",
        "StockOptionLevel",
        "PercentSalaryHike",
        "DailyRate",
        "HourlyRate",
        "MonthlyRate",
    ],
    "Career Progression": [
        "YearsSinceLastPromotion",
    ],
    "Workload & Travel": [
        "OverTimeFlag",
        "BusinessTravel",  # categorical control, see note below
    ],
    "Engagement & Satisfaction": [
        "JobInvolvement",
        "JobSatisfaction",
        "EnvironmentSatisfaction",
        "RelationshipSatisfaction",
        "WorkLifeBalance",
    ],
    "Commute": [
        "DistanceFromHome",
    ],
    "Mobility History": [
        "NumCompaniesWorked",
    ],
}

# Base controls for horse race (categorical only)
BASE_CONTROLS_CAT = ["Department", "JobRole", "JobLevel", "BusinessTravel", "MaritalStatus", "Gender"]
BASE_CONTROLS_NUM = []  # OverTimeFlag included in family; do not duplicate here

In [61]:
# Build horse race dataframe (hr_df) for fitting
# Use model_df if exists; if not, create it similarly
# model_df is created later, so create hr_df now based on df
# We create hr_df with needed columns plus controls

# Determine all needed columns for horse race:
hr_needed_cols = ["AttritionFlag"] + BASE_CONTROLS_CAT
# Add numeric candidates from all families except categorical-only ones
for fam_vars in FAMILIES.values():
    for v in fam_vars:
        if v not in hr_needed_cols and v in df.columns:
            hr_needed_cols.append(v)

hr_df = df[hr_needed_cols].dropna().copy()

# Create OverTimeFlag if not already in hr_df but in df (should be present)
if "OverTimeFlag" not in hr_df.columns and "OverTimeFlag" in df.columns:
    hr_df["OverTimeFlag"] = df["OverTimeFlag"]

# Outcome
y_hr = hr_df["AttritionFlag"].astype(int)

# Prepare base controls: categorical controls, one-hot encoded (drop_first=True)
cat_controls_in_hr = [c for c in BASE_CONTROLS_CAT if c in hr_df.columns]
X_base_cat = pd.get_dummies(hr_df[cat_controls_in_hr], drop_first=True)

# Standardize numeric controls (none here for base)
X_base_num = pd.DataFrame(index=hr_df.index)

# Combine base controls
X_base = pd.concat([X_base_cat, X_base_num], axis=1)

_seen_fit_errors = set()
# Helper functions for horse race fitting and selection

def standardize_series(s: pd.Series) -> pd.Series:
    mu = s.mean()
    sd = s.std(ddof=0)
    if sd == 0 or np.isnan(sd):
        return s
    return (s - mu) / sd

def fit_aic_for_terms(terms: list[str]) -> tuple[float, pd.Series, pd.Series]:
    """
    Fit logistic regression with base controls plus specified numeric terms.
    Returns (AIC, params, pvalues). If fit fails, returns (np.inf, empty_series, empty_series).

    Notes:
    - Forces numeric X
    - Drops NaN rows
    - Aligns y to X
    """
    try:
        X_terms = pd.DataFrame(index=hr_df.index)

        for term in terms:
            if term not in hr_df.columns:
                continue

            s = pd.to_numeric(hr_df[term], errors="coerce")
            s = standardize_series(s)
            X_terms[term] = s

        X_full = pd.concat([X_base, X_terms], axis=1)
        X_full = sm.add_constant(X_full, has_constant="add")

        # Force numeric; drop rows with NaNs; align
        X_full = X_full.apply(pd.to_numeric, errors="coerce")
        valid = X_full.notna().all(axis=1) & y_hr.notna()
        X_fit = X_full.loc[valid].astype(float)
        y_fit = y_hr.loc[valid].astype(int)

        if y_fit.nunique() < 2:
            empty_series = pd.Series(dtype=float)
            return (np.inf, empty_series, empty_series)

        res = sm.GLM(y_fit, X_fit, family=sm.families.Binomial()).fit()
        return (float(res.aic), res.params, res.pvalues)

    except Exception as e:
        key = (tuple(terms), type(e).__name__, str(e)[:120])
        if key not in _seen_fit_errors:
            _seen_fit_errors.add(key)
            print(f"[horse-race warn] Fit failed for terms={terms}: {type(e).__name__}: {e}")
        empty_series = pd.Series(dtype=float)
        return (np.inf, empty_series, empty_series)

selected_reps = {}
SELECTED_NUMERIC_DRIVERS = []

In [62]:
for family, candidates in FAMILIES.items():
    # Filter candidates to those present in hr_df columns or categorical controls
    candidates_present = [c for c in candidates if (c in hr_df.columns) or (c in BASE_CONTROLS_CAT)]
    if not candidates_present:
        # No candidates to test
        continue

    # Special handling for Workload & Travel family with BusinessTravel categorical control
    if family == "Workload & Travel":
        # BusinessTravel is a categorical control, already in base controls
        # Only OverTimeFlag is numeric candidate to test
        numeric_candidates = [c for c in candidates_present if c != "BusinessTravel"]
        if not numeric_candidates:
            # No numeric candidates to test; select none
            selected_reps[family] = []
            print(f"Note: In family '{family}', BusinessTravel is controlled categorically; no numeric candidates to test.")
            continue
        candidates_present = numeric_candidates
        print(f"Note: In family '{family}', BusinessTravel is controlled categorically; only numeric candidates tested.")

    # If only one candidate, select it directly
    if len(candidates_present) == 1:
        selected_reps[family] = candidates_present
        if candidates_present[0] not in BASE_CONTROLS_CAT:
            SELECTED_NUMERIC_DRIVERS.append(candidates_present[0])
        print(f"Family '{family}': Only one candidate '{candidates_present[0]}' present, selected by default.")
        continue

    # More than one candidate: run single-term horse race
    horse_race_results = []
    for cand in candidates_present:
        # Skip categorical controls in horse race numeric selection (except OverTimeFlag handled above)
        if cand in BASE_CONTROLS_CAT and family != "Workload & Travel":
            # Categorical controls not tested here
            continue
        aic, params, pvals = fit_aic_for_terms([cand])
        coef = params.get(cand, np.nan)
        pval = pvals.get(cand, np.nan)
        or_val = np.exp(coef) if not np.isnan(coef) else np.nan
        sign = "+" if coef >= 0 else "-"
        horse_race_results.append({
            "candidate": cand,
            "AIC": aic,
            "coef(sign)": f"{coef:.4f}({sign})" if not np.isnan(coef) else "NA",
            "OR": f"{or_val:.4f}" if not np.isnan(or_val) else "NA",
            "p_value": pval,
        })

    # Create DataFrame for display & sorting
    hr_df_results = pd.DataFrame(horse_race_results)
    hr_df_results = hr_df_results.sort_values("AIC", ascending=True).reset_index(drop=True)

    print(f"=== Horse Race Results for Family: {family} ===")
    print(hr_df_results.to_string(index=False))
    print()

    if hr_df_results.empty:
        # No candidates fit successfully
        selected_reps[family] = []
        continue

    # Select best candidate (lowest AIC)
    best_cand = hr_df_results.loc[0, "candidate"]
    selected_vars = [best_cand]
    if best_cand not in BASE_CONTROLS_CAT:
        SELECTED_NUMERIC_DRIVERS.append(best_cand)

    # Attempt to keep second variable if more than one candidate
    if len(candidates_present) > 1:
        remaining_cands = [c for c in candidates_present if c != best_cand and c not in BASE_CONTROLS_CAT]
        if remaining_cands:
            second_results = []
            # Fit base + best_cand + candidate, compute delta_AIC and p_value of candidate
            aic_best, _, _ = fit_aic_for_terms([best_cand])
            for cand2 in remaining_cands:
                aic_2, params_2, pvals_2 = fit_aic_for_terms([best_cand, cand2])
                delta_aic = aic_2 - aic_best
                pval_2 = pvals_2.get(cand2, np.nan)
                second_results.append({
                    "candidate": cand2,
                    "AIC": aic_2,
                    "delta_AIC": delta_aic,
                    "p_value": pval_2,
                    "coef": params_2.get(cand2, np.nan),
                })
            second_df = pd.DataFrame(second_results)
            second_df = second_df.sort_values("AIC", ascending=True).reset_index(drop=True)
            if not second_df.empty:
                best_second = second_df.loc[0]
                # Keep second if delta_AIC <= -2.0 and p_value < 0.10
                if (best_second["delta_AIC"] <= -2.0) and (best_second["p_value"] < 0.10):
                    selected_vars.append(best_second["candidate"])
                    if best_second["candidate"] not in BASE_CONTROLS_CAT:
                        SELECTED_NUMERIC_DRIVERS.append(best_second["candidate"])
                    print(f"Family '{family}': KEEP 2 variables based on delta_AIC <= -2.0 and p < 0.10 rule.")
                    print(f"Selected variables: {selected_vars}")
                else:
                    print(f"Family '{family}': KEEP 1 variable; second candidate did not meet delta_AIC and p-value criteria.")
                    print(f"Selected variable: {selected_vars}")
            else:
                print(f"Family '{family}': No viable second candidate found; KEEP 1 variable.")
                print(f"Selected variable: {selected_vars}")
        else:
            print(f"Family '{family}': Only one numeric candidate after filtering; KEEP 1 variable.")
            print(f"Selected variable: {selected_vars}")
    else:
        print(f"Family '{family}': Only one candidate; KEEP 1 variable.")
        print(f"Selected variable: {selected_vars}")

    selected_reps[family] = selected_vars
    print()

=== Horse Race Results for Family: Tenure & Experience ===
           candidate         AIC coef(sign)     OR  p_value
  YearsInCurrentRole 1169.975479 -0.3813(-) 0.6830 0.000121
YearsWithCurrManager 1170.732568 -0.3588(-) 0.6985 0.000173
      YearsAtCompany 1180.783197 -0.2503(-) 0.7786 0.028189
   TotalWorkingYears 1181.470794 -0.2878(-) 0.7499 0.040587

Family 'Tenure & Experience': KEEP 1 variable; second candidate did not meet delta_AIC and p-value criteria.
Selected variable: ['YearsInCurrentRole']

=== Horse Race Results for Family: Compensation & Rewards ===
        candidate         AIC coef(sign)     OR  p_value
        DailyRate 1182.976824 -0.1272(-) 0.8806 0.092303
 StockOptionLevel 1183.843978 -0.1650(-) 0.8479 0.166153
    MonthlyIncome 1185.063705  0.2725(+) 1.3132 0.383286
      MonthlyRate 1185.188925  0.0599(+) 1.0617 0.426608
PercentSalaryHike 1185.671323 -0.0294(-) 0.9710 0.699191
       HourlyRate 1185.820035  0.0027(+) 1.0027 0.971174

Family 'Compensation & Rew

In [63]:
# After selection, print summary
print("=== FAMILY SELECTION SUMMARY (for justification) ===")
for fam, vars_kept in selected_reps.items():
    print(f"{fam}: {vars_kept if vars_kept else 'No numeric drivers selected'}")
print()

=== FAMILY SELECTION SUMMARY (for justification) ===
Tenure & Experience: ['YearsInCurrentRole']
Compensation & Rewards: ['DailyRate']
Career Progression: ['YearsSinceLastPromotion']
Workload & Travel: ['OverTimeFlag']
Engagement & Satisfaction: ['JobInvolvement', 'JobSatisfaction']
Commute: ['DistanceFromHome']
Mobility History: ['NumCompaniesWorked']



In [64]:
# Ensure the core story variables are included (but do NOT re-add duplicates inside a family)
# Add Age explicitly (often highlighted in EDA) while keeping the within-family horse race intact.
always_include = [
    "Age",
    "OverTimeFlag",
    "YearsSinceLastPromotion",
    "DistanceFromHome",
    "NumCompaniesWorked",
]

for v in always_include:
    if v in df.columns and v not in SELECTED_NUMERIC_DRIVERS:
        SELECTED_NUMERIC_DRIVERS.append(v)

# Remove duplicates and sort
SELECTED_NUMERIC_DRIVERS = sorted(set(SELECTED_NUMERIC_DRIVERS))

print("Final driver set used in locked model:", SELECTED_NUMERIC_DRIVERS)
print()

# Update numeric_vars to selected representatives
if SELECTED_NUMERIC_DRIVERS:
    numeric_vars = SELECTED_NUMERIC_DRIVERS
# categorical_controls remain unchanged

Final driver set used in locked model: ['Age', 'DailyRate', 'DistanceFromHome', 'JobInvolvement', 'JobSatisfaction', 'NumCompaniesWorked', 'OverTimeFlag', 'YearsInCurrentRole', 'YearsSinceLastPromotion']



# Build Model

In [65]:
"""Build model matrix (controls + drivers)"""

# Build model frame (only columns we need) — UNIQUE list to avoid duplicate labels
use_cols_raw = ["AttritionFlag"]

# Ensure OverTimeFlag appears exactly once
if "OverTimeFlag" in df.columns:
    use_cols_raw.append("OverTimeFlag")

# Add numeric drivers excluding OverTimeFlag (already added)
use_cols_raw += [c for c in numeric_vars if c in df.columns and c != "OverTimeFlag"]

# Add categorical controls
use_cols_raw += [c for c in categorical_controls if c in df.columns]

# Order-preserving de-duplication
use_cols = list(dict.fromkeys(use_cols_raw))

model_df = df[use_cols].dropna().copy()

y = model_df["AttritionFlag"].astype(int)
X = model_df.drop(columns=["AttritionFlag"])  # includes OverTimeFlag


# Encode & Standardize Data

In [66]:
"""Encode categoricals + standardize numerics"""

# Collapse rare categories to reduce quasi-complete separation (improves convergence)
MIN_CAT_COUNT = 20
for c in categorical_controls:
    if c in X.columns:
        vc = X[c].value_counts(dropna=False)
        rare = vc[vc < MIN_CAT_COUNT].index
        if len(rare) > 0:
            X[c] = X[c].where(~X[c].isin(rare), other="Other")

# One-hot for categorical controls (drop_first avoids perfect multicollinearity)
cat_in_X = [c for c in categorical_controls if c in X.columns]
if cat_in_X:
    X = pd.get_dummies(X, columns=cat_in_X, drop_first=True)

# Standardize numeric-like columns for comparable odds ratios (per +1 SD)
# Note: if a label is duplicated, X[c] returns a DataFrame; we use the first occurrence.
numeric_like = [c for c in X.columns if c in (numeric_vars + ["OverTimeFlag"]) ]
for c in numeric_like:
    series = X[c]
    if isinstance(series, pd.DataFrame):
        print(f"[warn] Duplicate column label detected during standardization: '{c}'. Using first occurrence.")
        series = series.iloc[:, 0]

    mu = float(series.mean())
    sd = float(series.std(ddof=0))
    if sd == 0.0 or np.isnan(sd):
        continue

    X[c] = (series - mu) / sd

# Ensure all numeric
X = X.apply(pd.to_numeric, errors="raise").astype(float)
X = sm.add_constant(X)


# Fit Model

In [67]:
"""Fit statsmodels GLM (Binomial / logit)"""

res = sm.GLM(y, X, family=sm.families.Binomial()).fit()

# Present Results

In [68]:
"""Present results (executive + analyst views)"""

# Model Spec summary
print("=== MODEL SPECIFICATION ===")
print("Numeric drivers included:")
FRIENDLY_TERM_MAP = {
    "Age": "Age",
    "OverTimeFlag": "Overtime",
    "YearsSinceLastPromotion": "Promotion Delay",
    "YearsAtCompany": "Tenure",
    "YearsInCurrentRole": "Role Growth",
    "YearsWithCurrManager": "Manager Stability",
}
pretty_numeric = [FRIENDLY_TERM_MAP.get(v, v) for v in numeric_vars]
print(", ".join(pretty_numeric))
print("\nCategorical controls included:")
print(", ".join(categorical_controls))
print("\nExcluded variables (to avoid collinearity/duplicates):")
print("IncomeLevel, IncomeQuartile, PayRates, PromotionStage, TrainingBucket, AgeGroup (and other within-family duplicates)")
print("Constant columns dropped if present: Over18, EmployeeCount, StandardHours")
print()

# Baseline attrition rate and rows used
baseline_attr = model_df["AttritionFlag"].mean() * 100
print(f"Rows used in model: {len(model_df):,}")
print(f"Baseline attrition rate: {baseline_attr:.2f}%")
print()

# Full statsmodels summary
print("=== FULL MODEL SUMMARY (ANALYST VIEW) ===")
print(res.summary())
print()

=== MODEL SPECIFICATION ===
Numeric drivers included:
Age, DailyRate, DistanceFromHome, JobInvolvement, JobSatisfaction, NumCompaniesWorked, Overtime, Role Growth, Promotion Delay

Categorical controls included:
Department, JobRole, JobLevel, BusinessTravel, MaritalStatus, Gender

Excluded variables (to avoid collinearity/duplicates):
IncomeLevel, IncomeQuartile, PayRates, PromotionStage, TrainingBucket, AgeGroup (and other within-family duplicates)
Constant columns dropped if present: Over18, EmployeeCount, StandardHours

Rows used in model: 1,470
Baseline attrition rate: 16.12%

=== FULL MODEL SUMMARY (ANALYST VIEW) ===
                 Generalized Linear Model Regression Results                  
Dep. Variable:          AttritionFlag   No. Observations:                 1470
Model:                            GLM   Df Residuals:                     1441
Model Family:                Binomial   Df Model:                           28
Link Function:                  Logit   Scale:        

In [69]:
# Clean driver table
out = pd.DataFrame(
    {
        "term": res.params.index,
        "coef": res.params.values,
        "odds_ratio": np.exp(res.params.values),
        "p_value": res.pvalues.values,
    }
)
# Friendly names for key drivers (match deck wording)
FRIENDLY_TERM_MAP = {
    "Age": "Age",
    "OverTimeFlag": "Overtime",
    "YearsSinceLastPromotion": "Promotion Delay",
    "YearsAtCompany": "Tenure",
    "YearsInCurrentRole": "Role Growth",
    "YearsWithCurrManager": "Manager Stability",
}
# Apply exact-match renames first
out["term"] = out["term"].replace(FRIENDLY_TERM_MAP)
# Also handle common dummy-name patterns if they appear (leave others unchanged)
# (No-op unless those exact terms are present)

out = out[out["term"] != "const"].copy()
out["direction"] = np.where(out["odds_ratio"] >= 1.0, "Higher risk", "Lower risk")

def stars(p: float) -> str:
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""

out["sig"] = out["p_value"].apply(stars)
out = out.sort_values(["p_value", "odds_ratio"], ascending=[True, False])

show_cols = ["term", "direction", "odds_ratio", "p_value", "sig"]

print("=== TOP 5 TERMS BY STATISTICAL SIGNIFICANCE ===")
print(out[show_cols].head(5).to_string(index=False))
print()

sig_terms = out[out["p_value"] < 0.05]
print("=== STATISTICALLY SIGNIFICANT TERMS (p < 0.05) ===")
if sig_terms.empty:
    print("No terms are statistically significant at p < 0.05.")
else:
    print(sig_terms[show_cols].to_string(index=False))
print()


# Executive summary
print("=== EXECUTIVE INTERPRETATION ===")
print("Odds Ratios (OR) > 1 indicate factors associated with higher attrition risk.")
print("OR < 1 indicate factors associated with lower attrition risk.")
print("These are associations, not proven causations.")
print()


=== TOP 5 TERMS BY STATISTICAL SIGNIFICANCE ===
                term   direction  odds_ratio      p_value sig
            Overtime Higher risk    2.282991 2.569264e-23 ***
MaritalStatus_Single Higher risk    3.914588 1.022702e-07 ***
  NumCompaniesWorked Higher risk    1.555399 8.320787e-07 ***
      JobInvolvement  Lower risk    0.658380 8.615047e-07 ***
     JobSatisfaction  Lower risk    0.679983 5.693245e-06 ***

=== STATISTICALLY SIGNIFICANT TERMS (p < 0.05) ===
                            term   direction  odds_ratio      p_value sig
                        Overtime Higher risk    2.282991 2.569264e-23 ***
            MaritalStatus_Single Higher risk    3.914588 1.022702e-07 ***
              NumCompaniesWorked Higher risk    1.555399 8.320787e-07 ***
                  JobInvolvement  Lower risk    0.658380 8.615047e-07 ***
                 JobSatisfaction  Lower risk    0.679983 5.693245e-06 ***
BusinessTravel_Travel_Frequently Higher risk    6.001557 7.273575e-06 ***
          

# Save Output

In [70]:
# 1) Driver terms table (clean, deck/visuals-friendly)
driver_terms_path = os.path.join(OUTPUT_DIR, "logistic_regression_driver_terms.csv")
out.to_csv(driver_terms_path, index=False)

# 2) Small metadata file (what was included + baseline)
meta = {
    "rows_used": int(len(model_df)),
    "baseline_attrition_rate": float(model_df["AttritionFlag"].mean()),
    "numeric_drivers": [FRIENDLY_TERM_MAP.get(v, v) for v in numeric_vars],
    "categorical_controls": list(categorical_controls),
}
meta_path = os.path.join(OUTPUT_DIR, "logistic_regression_model_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] {driver_terms_path}")
print(f"[saved] {meta_path}")

[saved] /content/drive/MyDrive/550 Project/Regression Versions/simon/logistic_regression_driver_terms.csv
[saved] /content/drive/MyDrive/550 Project/Regression Versions/simon/logistic_regression_model_meta.json
